Genetic Algorithm

Helper Functions

In [ ]:
import numpy as np
import pandas as pd

def top_k(arr, k):
    return np.argpartition(arr, -k)[-k:]

def elitism(e, population, fitnesses):
    top = top_k(fitnesses, e)
    return population[top]

class Selector(object):
    def __init__(self):
        super().__init__()

    def prep_selector(self, fitnesses):
        pass

    def select(self, population):
        pass


class RouletteWheelSelector(Selector):
    def __init__(self):
        super().__init__()
        self.table = []

    def prep_selector(self, fitnesses):
        transformed = np.exp(fitnesses)
        total = np.sum(transformed)
        self.table = transformed / total
        self.indicies = np.arange(0, len(fitnesses), 1, dtype=int)

    def select(self, population):
        parent1_idx, parent2_idx = np.random.choice(self.indicies, size=2, replace=False, p=self.table)
        parent1 = population[parent1_idx, :]
        parent2 = population[parent2_idx, :]
        return parent1, parent2

class TournamentSelection(Selector):
    def __init__(self):
        super().__init__()
        self.table = []

    def prep_selector(self, fitnesses):
        self.table = fitnesses

    def select(self,population):
        indicies = np.arange(0, len(population), 1, dtype=int)
        sub_population = population[indicies]
        sub_pop_fitnesses = self.table[indicies]
        parent1_idx, parent2_idx = top_k(sub_pop_fitnesses, 2)
        return sub_population[parent1_idx], sub_population[parent2_idx]


def single_point(parent1, parent2):
    point = np.random.randint(1, len(parent1))
    child = np.zeros_like(parent1)
    child[:point] = parent1[:point]
    child[point:] = parent2[point:]
    return child


def interpolation(parent1, parent2):
    lambda_point = np.random.uniform()
    return lambda_point * parent1 +  (1 - lambda_point) * parent2

def uniform(parent1, parent2):
    selection = np.random.randint(0, 2, size=len(parent1))
    neg_selection = 1 - selection
    child = parent1 * selection + parent2 * neg_selection
    return child

class Company:
    def __init__(self, name):
        self.name = name
        self.sales2024 = 0
        self.sales2026 = 0
        self.games_2024 = []
        self.games_2026 = []

    def add_game(self, game_record, year):
        """Add a game record to the correct year and accumulate sales."""
        total_sales = (
            game_record['NA_Sales'] +
            game_record['EU_Sales'] +
            game_record['JP_Sales'] +
            game_record['Other_Sales']
        )

        if year == 2024:
            self.games_2024.append(game_record)
            self.sales2024 += total_sales
        elif year == 2026:
            self.games_2026.append(game_record)
            self.sales2026 += total_sales

    @property
    def profit(self):
        return self.sales2026 - self.sales2024

    def __repr__(self):
        return (f"Company(name={self.name}, "
                f"sales2024={self.sales2024:.2f}, "
                f"sales2026={self.sales2026:.2f}, "
                f"profit={self.profit:.2f})")

def load_data(df, companies, year):
    """Load a dataframe into the companies dict for a given year."""
    for _, row in df.iterrows():
        publisher = row['Publisher']

        if publisher not in companies:
            companies[publisher] = Company(name=publisher)

        game_record = {
            'title':        row['Name'],
            'platform':     row['Platform'],
            'genre':        row['Genre'],
            'release_year': row['Release_Year'],
            'NA_Sales':     row['NA_Sales'],
            'EU_Sales':     row['EU_Sales'],
            'JP_Sales':     row['JP_Sales'],
            'Other_Sales':  row['Other_Sales'],
        }

        companies[publisher].add_game(game_record, year)

Random Data Sampling for testing

In [ ]:
df_2024 = pd.read_csv("../data/vgData2024.csv")
df_2026 = pd.read_csv("../data/vgData2026.csv")

df_2024_RS=df_2024.sample(n=1000, random_state=250)
df_2026_RS=df_2026.sample(n=1000, random_state=250)
#With both datasets having the same entires, the same seed would return the exact same entires in both datasets

companies = {}

load_data(df_2024_RS, companies, year=2024)
load_data(df_2026_RS, companies, year=2026)

for company in companies.values():
    if(company.profit > 0):
        print(company)

Agent Code

In [ ]:
'''class Agent:
    def __init__(self, dna=None):
        #self.dna = dna if dna is None else'''